# Coupled Neutron-Photon Model - ENDF/B-VIII.1

This notebook compares YAMC and OpenMC simulation results for coupled neutron-photon transport using a broomstick (cylinder) geometry with the ENDF/B-VIII.1 nuclear data library.

## Model Description

- **Geometry**: Cylinder with radius 1 cm, height 100 cm (z = -50 to +50), vacuum boundaries
- **Source**: Point source at origin, 14.06 MeV monoenergetic neutrons, isotropic
- **Material**: Single nuclide at 1.0 g/cm³, temperature 294 K
- **Transport**: Coupled neutron-photon (`photon_transport=True`)
- **Statistics**: 500,000 particles × 10 batches = 5,000,000 total histories
- **Tallies**: Neutron flux (VITAMIN-J-175), photon flux (VITAMIN-J-42), neutron reactions (per-MT), photon interactions

In [1]:
# Setup and imports
import re
import numpy as np
import pandas as pd
import glob
import os
import yamc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
NUCLEAR_DATA_DIR = {
    "endf-b8.1": "/home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/",
}

NUCLEAR_DATA = "endf-b8.1"
PARTICLES = 500000
BATCHES = 10
SOURCE_ENERGY = 14.06e6  # 14.06 MeV neutrons
FLUX_DECADES = 6

# Cylinder dimensions
CYLINDER_RADIUS = 1.0  # cm
CYLINDER_HALF_HEIGHT = 50.0  # cm (total height = 100 cm)

# Group structures
NEUTRON_GROUP_STRUCTURE = "VITAMIN-J-175"
PHOTON_GROUP_STRUCTURE = "VITAMIN-J-42"

# Neutron scalar scores
NEUTRON_SCALAR_SCORES = [
    "flux", "heating", "heating-local", "total", "absorption",
    "H1-production", "H2-production", "H3-production",
    "He3-production", "He4-production",
]

# Photon scalar scores
PHOTON_SCALAR_SCORES = [
    "flux", "coherent-scatter", "incoherent-scatter",
    "photoelectric", "pair-production", "heating",
]

# Score-to-MT mapping for deduplication
SCORE_TO_MT = {
    "total": 1, "absorption": 27, "heating": 301, "heating-local": 901,
    "H1-production": 203, "H2-production": 204, "H3-production": 205,
    "He3-production": 206, "He4-production": 207,
    "coherent-scatter": 502, "incoherent-scatter": 504,
    "photoelectric": 522, "pair-production": 516,
}

In [2]:
def run_openmc_simulation(nuclide, neutron_path, element, photon_path):
    """Run OpenMC coupled neutron-photon simulation for a single nuclide."""
    import openmc as mc

    mc.reset_auto_ids()
    mc.config['cross_sections'] = f"{NUCLEAR_DATA_DIR[NUCLEAR_DATA]}cross_sections.xml"

    # Geometry: broomstick cylinder
    cylinder = mc.ZCylinder(r=CYLINDER_RADIUS, boundary_type='vacuum')
    z_bot = mc.ZPlane(z0=-CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    z_top = mc.ZPlane(z0=CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    region = -cylinder & +z_bot & -z_top

    # Material
    material = mc.Material()
    material.temperature = 294
    material.add_nuclide(nuclide, 1.0)
    material.set_density("g/cm3", 1.0)

    cell = mc.Cell(name="broomstick", region=region, fill=material)
    geometry = mc.Geometry([cell])

    # Source: 14.06 MeV isotropic neutrons at origin
    source = mc.IndependentSource(
        space=mc.stats.Point([0, 0, 0]),
        energy=mc.stats.Discrete([SOURCE_ENERGY], [1.0]),
        angle=mc.stats.Isotropic(),
    )

    settings = mc.Settings(
        particles=PARTICLES,
        batches=BATCHES,
        source=source,
        seed=1,
        run_mode='fixed source',
        output={'tallies': False, 'summary': False},
    )
    settings.photon_transport = True

    # Filters and tallies
    cell_filter = mc.CellFilter(cell)
    neutron_filter = mc.ParticleFilter("neutron")
    photon_filter = mc.ParticleFilter("photon")
    tallies = []

    # Neutron scalar scores
    for score in NEUTRON_SCALAR_SCORES:
        t = mc.Tally()
        t.filters = [cell_filter, neutron_filter]
        t.scores = [score]
        t.name = f"n_{score}"
        tallies.append(t)

    # Per-MT neutron reactions
    in_neu = mc.data.IncidentNeutron.from_hdf5(neutron_path)
    named_mts = set(SCORE_TO_MT.values())
    mt_list = sorted(in_neu.reactions.keys())
    for mt in mt_list:
        if mt in named_mts:
            continue
        t = mc.Tally()
        t.filters = [cell_filter, neutron_filter]
        t.scores = [str(mt)]
        t.name = str(mt)
        tallies.append(t)

    # Photon scalar scores
    for score in PHOTON_SCALAR_SCORES:
        t = mc.Tally()
        t.filters = [cell_filter, photon_filter]
        t.scores = [score]
        t.name = f"p_{score}"
        tallies.append(t)

    # Neutron flux spectrum
    n_energy_filter = mc.EnergyFilter.from_group_structure(NEUTRON_GROUP_STRUCTURE)
    n_bins_2d = n_energy_filter.bins
    n_energy_bins = np.concatenate([[n_bins_2d[0, 0]], n_bins_2d[:, 1]])
    t_n_spec = mc.Tally()
    t_n_spec.filters = [cell_filter, neutron_filter, n_energy_filter]
    t_n_spec.scores = ["flux"]
    t_n_spec.name = "neutron_spectrum"
    tallies.append(t_n_spec)

    # Photon flux spectrum
    p_energy_filter = mc.EnergyFilter.from_group_structure(PHOTON_GROUP_STRUCTURE)
    p_bins_2d = p_energy_filter.bins
    p_energy_bins = np.concatenate([[p_bins_2d[0, 0]], p_bins_2d[:, 1]])
    t_p_spec = mc.Tally()
    t_p_spec.filters = [cell_filter, photon_filter, p_energy_filter]
    t_p_spec.scores = ["flux"]
    t_p_spec.name = "photon_spectrum"
    tallies.append(t_p_spec)

    # Run
    model = mc.Model(geometry=geometry, settings=settings, tallies=tallies)
    sp_filename = model.run(apply_tally_results=True)
    sp = mc.StatePoint(sp_filename)

    transport_time = sp.runtime['transport']
    total_particles = PARTICLES * BATCHES
    particles_per_second = total_particles / transport_time

    # Collect results
    n_scalars = {}
    n_stds = {}
    p_scalars = {}
    p_stds = {}
    mt_scalars = {}
    mt_stds = {}

    for t in tallies:
        if t.name in ("neutron_spectrum", "photon_spectrum"):
            continue
        val = float(t.mean.flatten()[0])
        std = float(t.std_dev.flatten()[0])
        if t.name.startswith("n_"):
            n_scalars[t.name[2:]] = val
            n_stds[t.name[2:]] = std
        elif t.name.startswith("p_"):
            p_scalars[t.name[2:]] = val
            p_stds[t.name[2:]] = std
        else:
            mt_scalars[t.name] = val
            mt_stds[t.name] = std

    n_scalars.update(mt_scalars)
    n_stds.update(mt_stds)

    return {
        'nuclide': nuclide,
        'code': 'openmc',
        'particles/s': particles_per_second,
        'n_scalars': n_scalars,
        'n_stds': n_stds,
        'p_scalars': p_scalars,
        'p_stds': p_stds,
        'n_spectrum_mean': np.array(t_n_spec.mean).squeeze(),
        'n_spectrum_std': np.array(t_n_spec.std_dev).squeeze(),
        'n_energy_bins': n_energy_bins,
        'p_spectrum_mean': np.array(t_p_spec.mean).squeeze(),
        'p_spectrum_std': np.array(t_p_spec.std_dev).squeeze(),
        'p_energy_bins': p_energy_bins,
    }

In [3]:
def run_yamc_simulation(nuclide, neutron_path, element, photon_path):
    """Run YAMC coupled neutron-photon simulation for a single nuclide. Returns (results_dict, model)."""
    import yamc as mc

    # Geometry: broomstick cylinder
    cylinder = mc.ZCylinder(radius=CYLINDER_RADIUS, boundary='vacuum')
    z_bot = mc.ZPlane(z0=-CYLINDER_HALF_HEIGHT, boundary='vacuum')
    z_top = mc.ZPlane(z0=CYLINDER_HALF_HEIGHT, boundary='vacuum')
    region = -cylinder & +z_bot & -z_top

    # Material with both neutron and photon data
    material = mc.Material(
        composition={nuclide: 1.0},
        density=1.0,
        temperature=294,
        id=1,
    )
    material.read_nuclear_data('endf-b8.1')

    cell = mc.Cell(id=1, name="broomstick", region=region, material=material)
    geometry = mc.Geometry([cell])

    # Source: 14.06 MeV isotropic neutrons at origin
    source = mc.NeutronSource(
        position=mc.PointSource(0, 0, 0),
        direction=mc.Isotropic(),
        energy=mc.Discrete([SOURCE_ENERGY], [1.0]),
    )

    tallies = []

    # Neutron scalar scores
    for score in NEUTRON_SCALAR_SCORES:
        t = mc.Tally(
            scores=[str(score)],
            name=f"n_{score}",
            cell=cell,
            particle="neutron",
        )
        tallies.append(t)

    # Per-MT neutron reactions
    named_mts = set(SCORE_TO_MT.values())
    mt_list = material.reaction_mts
    for mt in mt_list:
        if mt in named_mts:
            continue
        t = mc.Tally(
            scores=[str(mt)],
            name=str(mt),
            cell=cell,
            particle="neutron",
        )
        tallies.append(t)

    # Photon scalar scores
    for score in PHOTON_SCALAR_SCORES:
        t = mc.Tally(
            scores=[str(score)],
            name=f"p_{score}",
            cell=cell,
            particle="photon",
        )
        tallies.append(t)

    # Neutron flux spectrum
    t_n_spec = mc.Tally(
        scores=["flux"],
        name="neutron_spectrum",
        cell=cell,
        particle="neutron",
        energy_group_structure=NEUTRON_GROUP_STRUCTURE,
    )
    tallies.append(t_n_spec)
    n_energy_bins = np.array(t_n_spec.energy_bins)

    # Photon flux spectrum
    t_p_spec = mc.Tally(
        scores=["flux"],
        name="photon_spectrum",
        cell=cell,
        particle="photon",
        energy_group_structure=PHOTON_GROUP_STRUCTURE,
    )
    tallies.append(t_p_spec)
    p_energy_bins = np.array(t_p_spec.energy_bins)

    # Run
    model = mc.Model(
        geometry=geometry,
        tallies=tallies,
        source=source,
        particles_per_batch=PARTICLES,
        batches=BATCHES,
        seed=1,
        photon_transport=True,
    )
    sim_results = model.simulate()

    # Collect results
    n_scalars = {}
    n_stds = {}
    p_scalars = {}
    p_stds = {}
    mt_scalars = {}
    mt_stds = {}

    for t in tallies:
        if t.name in ("neutron_spectrum", "photon_spectrum"):
            continue
        r = sim_results[t]
        val = r.mean[0]
        std = r.standard_deviation[0]
        if t.name.startswith("n_"):
            n_scalars[t.name[2:]] = val
            n_stds[t.name[2:]] = std
        elif t.name.startswith("p_"):
            p_scalars[t.name[2:]] = val
            p_stds[t.name[2:]] = std
        else:
            mt_scalars[t.name] = val
            mt_stds[t.name] = std

    n_scalars.update(mt_scalars)
    n_stds.update(mt_stds)

    n_spec = sim_results[t_n_spec]
    p_spec = sim_results[t_p_spec]

    return {
        'nuclide': nuclide,
        'code': 'yamc',
        'particles/s': model.particles_per_second,
        'n_scalars': n_scalars,
        'n_stds': n_stds,
        'p_scalars': p_scalars,
        'p_stds': p_stds,
        'n_spectrum_mean': np.array(n_spec.mean).squeeze(),
        'n_spectrum_std': np.array(n_spec.standard_deviation).squeeze(),
        'n_energy_bins': n_energy_bins,
        'p_spectrum_mean': np.array(p_spec.mean).squeeze(),
        'p_spectrum_std': np.array(p_spec.standard_deviation).squeeze(),
        'p_energy_bins': p_energy_bins,
    }, model

In [4]:
def create_step_trace(energy_bins, values, name, color, dash=None):
    """Create a step plot trace for Plotly (similar to matplotlib step with where='post')."""
    x_step = []
    y_step = []
    for i in range(len(values)):
        x_step.extend([energy_bins[i], energy_bins[i+1]])
        y_step.extend([values[i], values[i]])

    return go.Scatter(
        x=x_step,
        y=y_step,
        mode='lines',
        name=name,
        line=dict(color=color, width=2, dash=dash),
    )


def create_comparison_plot(energy_bins, openmc_flux, openmc_std, yamc_flux, yamc_std, title_prefix="Flux"):
    """Create a Plotly subplot with flux spectra (top) and ratio (bottom)."""

    energy_bins = np.asarray(energy_bins)
    bin_centers = np.sqrt(energy_bins[:-1] * energy_bins[1:])

    omc_flux = openmc_flux.copy()
    omc_std = openmc_std.copy()
    ymc_flux = yamc_flux.copy()
    ymc_std = yamc_std.copy()

    # Normalize by lethargy bin width
    lethargy_bin_width = np.log10(energy_bins[1:] / energy_bins[:-1])
    omc_flux = omc_flux / lethargy_bin_width
    omc_std = omc_std / lethargy_bin_width
    ymc_flux = ymc_flux / lethargy_bin_width
    ymc_std = ymc_std / lethargy_bin_width

    # Find the lowest energy bin with non-zero flux
    has_flux = (omc_flux > 0) | (ymc_flux > 0)
    if np.any(has_flux):
        first_nonzero_idx = np.argmax(has_flux)
        min_plot_energy = energy_bins[first_nonzero_idx]
    else:
        min_plot_energy = energy_bins[0]

    max_plot_energy = energy_bins[-1] * 1.1

    max_flux = max(
        np.max(omc_flux[omc_flux > 0]) if np.any(omc_flux > 0) else 1e-10,
        np.max(ymc_flux[ymc_flux > 0]) if np.any(ymc_flux > 0) else 1e-10
    )
    flux_threshold = max_flux / (10 ** FLUX_DECADES)

    omc_flux_plot = np.where(omc_flux >= flux_threshold, omc_flux, np.nan)
    ymc_flux_plot = np.where(ymc_flux >= flux_threshold, ymc_flux, np.nan)
    omc_std_plot = np.where(omc_flux >= flux_threshold, omc_std, np.nan)
    ymc_std_plot = np.where(ymc_flux >= flux_threshold, ymc_std, np.nan)

    # Calculate ratio with error propagation
    valid = (omc_flux > 0) & (ymc_flux > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(valid, ymc_flux / omc_flux, np.nan)
        rel_err_yamc = np.where(ymc_flux > 0, ymc_std / ymc_flux, 0)
        rel_err_openmc = np.where(omc_flux > 0, omc_std / omc_flux, 0)
    ratio_rel_err = np.sqrt(rel_err_yamc**2 + rel_err_openmc**2)
    ratio_std = np.where(valid, ratio * ratio_rel_err, np.nan)

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.7, 0.3],
        subplot_titles=(f'{title_prefix} per Unit Lethargy', 'YAMC / OpenMC Ratio')
    )

    fig.add_trace(
        create_step_trace(energy_bins, omc_flux_plot, 'OpenMC', 'blue'),
        row=1, col=1
    )
    fig.add_trace(
        create_step_trace(energy_bins, ymc_flux_plot, 'YAMC', 'red', dash='dash'),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=bin_centers, y=omc_flux_plot, mode='markers',
            name='OpenMC error',
            marker=dict(color='blue', size=4, symbol='circle'),
            error_y=dict(type='data', array=omc_std_plot, visible=True, color='blue', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=bin_centers, y=ymc_flux_plot, mode='markers',
            name='YAMC error',
            marker=dict(color='red', size=4, symbol='square'),
            error_y=dict(type='data', array=ymc_std_plot, visible=True, color='red', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )

    fig.add_trace(
        create_step_trace(energy_bins, ratio, 'YAMC/OpenMC', 'green'),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([bin_centers, bin_centers[::-1]]),
            y=np.concatenate([ratio + ratio_std, (ratio - ratio_std)[::-1]]),
            fill='toself',
            fillcolor='rgba(0, 128, 0, 0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            name='Uncertainty',
            showlegend=False,
        ),
        row=2, col=1
    )
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=1)

    fig.update_xaxes(type="log", title_text="Energy (eV)",
                     range=[np.log10(min_plot_energy), np.log10(max_plot_energy)],
                     exponentformat='e', row=2, col=1)
    fig.update_xaxes(type="log",
                     range=[np.log10(min_plot_energy), np.log10(max_plot_energy)],
                     exponentformat='e', row=1, col=1)
    fig.update_yaxes(type="log", title_text="Flux / Lethargy", exponentformat='e', row=1, col=1)
    fig.update_yaxes(title_text="Ratio", range=[0.5, 1.5], row=2, col=1)

    fig.update_layout(
        height=700,
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=60, r=20, t=40, b=40)
    )

    return fig


def calculate_spectral_agreement(openmc_flux, yamc_flux, openmc_std, yamc_std, flux_threshold=1e-6):
    """Calculate statistical agreement metrics for flux spectra comparison."""

    chi_squared_terms = []
    rel_diffs = []
    weighted_rel_diffs = []
    flux_weights = []

    for i in range(len(openmc_flux)):
        omc = openmc_flux[i]
        ymc = yamc_flux[i]
        omc_std = openmc_std[i]
        ymc_std = yamc_std[i]

        combined_std = np.sqrt(omc_std**2 + ymc_std**2)
        if combined_std > 0:
            chi_squared_terms.append(((ymc - omc) / combined_std)**2)

        if omc > flux_threshold or ymc > flux_threshold:
            if omc > 0:
                rel_diff = abs((ymc - omc) / omc) * 100
                if np.isfinite(rel_diff):
                    rel_diffs.append(rel_diff)
                    weight = max(omc, ymc)
                    weighted_rel_diffs.append(rel_diff * weight)
                    flux_weights.append(weight)

    chi_squared = np.sum(chi_squared_terms)
    dof = len(chi_squared_terms)
    reduced_chi_squared = chi_squared / dof if dof > 0 else 0.0

    median_rel_diff = np.median(rel_diffs) if rel_diffs else 0.0
    weighted_avg_rel_diff = np.sum(weighted_rel_diffs) / np.sum(flux_weights) if flux_weights else 0.0

    if reduced_chi_squared < 1.5:
        quality = "Excellent"
    elif reduced_chi_squared < 3.0:
        quality = "Good"
    elif reduced_chi_squared < 5.0:
        quality = "Fair"
    else:
        quality = "Poor"

    return {
        'reduced_chi_squared': reduced_chi_squared,
        'degrees_of_freedom': dof,
        'median_rel_diff': median_rel_diff,
        'weighted_avg_rel_diff': weighted_avg_rel_diff,
        'quality': quality,
        'bins_analyzed': len(rel_diffs),
        'total_bins': len(openmc_flux)
    }


def create_agreement_table(stats):
    """Create a summary table of spectral agreement statistics."""
    rows = [
        {'Metric': 'Reduced Chi-Squared', 'Value': f"{stats['reduced_chi_squared']:.2f}",
         'Note': f"(ideal ~ 1.0, dof={stats['degrees_of_freedom']})"},
        {'Metric': 'Weighted Avg Rel. Diff', 'Value': f"{stats['weighted_avg_rel_diff']:.2f}%",
         'Note': '(flux-weighted)'},
        {'Metric': 'Median Rel. Diff', 'Value': f"{stats['median_rel_diff']:.2f}%",
         'Note': '(robust to outliers)'},
        {'Metric': 'Agreement Quality', 'Value': stats['quality'],
         'Note': f"({stats['bins_analyzed']}/{stats['total_bins']} bins analyzed)"},
    ]
    return pd.DataFrame(rows)


# MT number to reaction description mapping from YAMC
MT_DESCRIPTIONS = yamc.reaction_names()

# Photon interaction MT descriptions
PHOTON_MT_DESCRIPTIONS = {
    'flux': 'Total photon flux',
    'coherent-scatter': 'Coherent (Rayleigh) scatter (MT 502)',
    'incoherent-scatter': 'Incoherent (Compton) scatter (MT 504)',
    'photoelectric': 'Photoelectric absorption (MT 522)',
    'pair-production': 'Pair production (MT 516)',
    'heating': 'Heating (KERMA)',
}


def get_mt_and_reaction(score_name):
    """Get MT number and reaction description for a score."""
    try:
        mt = int(score_name)
        return str(mt), MT_DESCRIPTIONS.get(mt, f'MT {mt}')
    except ValueError:
        pass

    if score_name in SCORE_TO_MT:
        mt = SCORE_TO_MT[score_name]
        return str(mt), score_name

    return '-', score_name


def create_neutron_tally_table(openmc_results, yamc_results):
    """Create a comparison table of neutron tallies, deduplicated by MT."""

    all_tallies = set(openmc_results['n_scalars'].keys()) | set(yamc_results['n_scalars'].keys())

    mt_to_scores = {}
    for name in all_tallies:
        mt_display, _ = get_mt_and_reaction(name)
        if mt_display not in mt_to_scores:
            mt_to_scores[mt_display] = []
        mt_to_scores[mt_display].append(name)

    rows = []

    for mt_display, score_names in mt_to_scores.items():
        best_openmc_val = np.nan
        best_openmc_std = np.nan
        best_yamc_val = np.nan
        best_yamc_std = np.nan
        best_reaction = None

        for name in score_names:
            openmc_val = openmc_results['n_scalars'].get(name, np.nan)
            openmc_std = openmc_results['n_stds'].get(name, np.nan)
            yamc_val = yamc_results['n_scalars'].get(name, np.nan)
            yamc_std = yamc_results['n_stds'].get(name, np.nan)
            _, reaction = get_mt_and_reaction(name)

            if not np.isnan(openmc_val) and openmc_val != 0:
                if np.isnan(best_openmc_val) or best_openmc_val == 0:
                    best_openmc_val = openmc_val
                    best_openmc_std = openmc_std
            elif np.isnan(best_openmc_val):
                best_openmc_val = openmc_val
                best_openmc_std = openmc_std

            if not np.isnan(yamc_val) and yamc_val != 0:
                if np.isnan(best_yamc_val) or best_yamc_val == 0:
                    best_yamc_val = yamc_val
                    best_yamc_std = yamc_std
            elif np.isnan(best_yamc_val):
                best_yamc_val = yamc_val
                best_yamc_std = yamc_std

            if best_reaction is None:
                best_reaction = reaction

        if mt_display != '-':
            try:
                mt_num = int(mt_display)
                best_reaction = MT_DESCRIPTIONS.get(mt_num, best_reaction)
            except ValueError:
                pass

        openmc_is_zero = best_openmc_val == 0 or np.isnan(best_openmc_val)
        yamc_is_zero = best_yamc_val == 0 or np.isnan(best_yamc_val)
        if openmc_is_zero and yamc_is_zero:
            continue

        if best_openmc_val != 0 and not np.isnan(best_openmc_val):
            rel_diff = (best_yamc_val - best_openmc_val) / best_openmc_val * 100
        else:
            rel_diff = np.nan

        rows.append({
            'MT': mt_display,
            'Reaction': best_reaction,
            'OpenMC': f"{best_openmc_val:.4e} +/- {best_openmc_std:.2e}",
            'YAMC': f"{best_yamc_val:.4e} +/- {best_yamc_std:.2e}",
            'Rel. Diff (%)': rel_diff,
            '_mt_sort': int(mt_display) if mt_display != '-' else 9999
        })

    rows.sort(key=lambda x: x['_mt_sort'])

    for row in rows:
        del row['_mt_sort']
        if not np.isnan(row['Rel. Diff (%)']):
            row['Rel. Diff (%)'] = f"{row['Rel. Diff (%)']:.2f}%"
        else:
            row['Rel. Diff (%)'] = 'N/A'

    return pd.DataFrame(rows)


def create_photon_tally_table(openmc_results, yamc_results):
    """Create a comparison table of photon tallies."""

    all_tallies = set(openmc_results['p_scalars'].keys()) | set(yamc_results['p_scalars'].keys())
    rows = []

    for name in sorted(all_tallies):
        openmc_val = openmc_results['p_scalars'].get(name, np.nan)
        openmc_std = openmc_results['p_stds'].get(name, np.nan)
        yamc_val = yamc_results['p_scalars'].get(name, np.nan)
        yamc_std = yamc_results['p_stds'].get(name, np.nan)

        if (openmc_val == 0 or np.isnan(openmc_val)) and (yamc_val == 0 or np.isnan(yamc_val)):
            continue

        if openmc_val != 0 and not np.isnan(openmc_val):
            rel_diff = (yamc_val - openmc_val) / openmc_val * 100
        else:
            rel_diff = np.nan

        reaction = PHOTON_MT_DESCRIPTIONS.get(name, name)

        rows.append({
            'Score': name,
            'Reaction': reaction,
            'OpenMC': f"{openmc_val:.4e} +/- {openmc_std:.2e}",
            'YAMC': f"{yamc_val:.4e} +/- {yamc_std:.2e}",
            'Rel. Diff (%)': f"{rel_diff:.2f}%" if not np.isnan(rel_diff) else 'N/A',
        })

    return pd.DataFrame(rows)

In [5]:
# Discover nuclides with both neutron and photon data
neutron_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'neutron')
photon_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'photon')

def element_from_nuclide(nuclide):
    """Extract element symbol from nuclide name: 'Fe56' -> 'Fe'."""
    return re.sub(r"\d+", "", nuclide)

h5_files = sorted(glob.glob(os.path.join(neutron_dir, '*.h5')))
nuclides = []
for f in h5_files:
    name = os.path.splitext(os.path.basename(f))[0]
    if '_m' in name:  # skip metastable
        continue
    element = element_from_nuclide(name)
    photon_path = os.path.join(photon_dir, f"{element}.h5")
    if os.path.exists(photon_path):
        nuclides.append((name, f, element, photon_path))

if not nuclides:
    raise FileNotFoundError(
        f"No coupled nuclides found in {neutron_dir}\n"
        f"Please check that neutron/ and photon/ subdirectories exist."
    )

print(f"Running coupled simulations for {len(nuclides)} nuclides...")
for nuc, _, elem, _ in nuclides:
    print(f"  {nuc} ({elem})")

Running coupled simulations for 5 nuclides...
  Fe54 (Fe)
  Fe55 (Fe)
  Fe56 (Fe)
  Fe57 (Fe)
  Fe58 (Fe)


In [6]:
# Run all simulations and collect results
all_results = {}
first_yamc_model = None

for nuclide, neutron_path, element, photon_path in nuclides:
    print(f"\nProcessing {nuclide} ({element})...")
    try:
        yamc_results, yamc_model = run_yamc_simulation(nuclide, neutron_path, element, photon_path)
        if first_yamc_model is None:
            first_yamc_model = yamc_model
    except Exception as e:
        print(f"  YAMC Error: {e}")
        continue

    try:
        openmc_results = run_openmc_simulation(nuclide, neutron_path, element, photon_path)
        all_results[nuclide] = {
            'openmc': openmc_results,
            'yamc': yamc_results,
            'element': element,
        }
        print(f"  OpenMC: {openmc_results['particles/s']:.0f} particles/s")
        print(f"  YAMC:   {yamc_results['particles/s']:.0f} particles/s")
    except Exception as e:
        print(f"  OpenMC Error: {e}")
        continue

print(f"\nCompleted {len(all_results)} / {len(nuclides)} nuclides successfully.")


Processing Fe54 (Fe)...


  Loaded photon data (Arrow): Z=26, 7701 energy points, 10 subshells, 7 Compton shells
Starting batch 1/10


Starting batch 2/10


Starting batch 3/10


Starting batch 4/10


Starting batch 5/10


... (85 outputs left out)

 Simulating batch 8


 Simulating batch 9


 Simulating batch 10


 Creating state point statepoint.10.h5...

 =======================>     TIMING STATISTICS     <=======================

 Total time for initialization     = 9.6468e-02 seconds
   Reading cross sections          = 9.3872e-02 seconds
 Total time in simulation          = 2.5324e+00 seconds
   Time in transport only          = 2.2605e+00 seconds
   Time in active batches          = 2.5324e+00 seconds
   Time accumulating tallies       = 2.4463e-01 seconds
   Time writing statepoints        = 2.6334e-02 seconds
 Total time for finalization       = 1.6000e-07 seconds
 Total time elapsed                = 2.6290e+00 seconds
 Calculation Rate (active)         = 1.97444e+06 particles/second

 ============================>     RESULTS     <============================

 Leakage Fraction            = 1.05841 +/- 0.00011

  OpenMC: 2211912 particles/s
  YAMC:   2131247 particles/s

Completed 5 / 5 nuclides successfully.


In [7]:
# Display broomstick model geometry with source from first simulation
import base64
from IPython.display import HTML

if first_yamc_model is not None:
    html_plot = first_yamc_model.plot(basis="xz", n_samples=500, axis_units="cm")
    html_plot.save("coupled_model_plot.html")
    encoded = base64.b64encode(html_plot.html.encode()).decode()
    display(HTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="600" frameborder="0"></iframe>'))
else:
    print("No successful YAMC simulation to plot.")

/home/jon/yamc-org/yamc-verification/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Results by Nuclide

Below are the comparison results for each nuclide. Each section includes:

- Interactive neutron and photon flux spectra plots (log-log scale)
- YAMC/OpenMC ratio plots with uncertainty bands
- Neutron tally comparison table (MT-sorted)
- Photon tally comparison table
- Spectral agreement statistics for both particle types
- Performance comparison (particles/second)

In [8]:
# Display results for each nuclide
for nuclide, results in all_results.items():
    openmc_results = results['openmc']
    yamc_results = results['yamc']
    element = results['element']

    # Header
    display(Markdown(f"## {nuclide} ({element})"))

    # Performance comparison
    openmc_pps = openmc_results['particles/s']
    yamc_pps = yamc_results['particles/s']
    speedup = yamc_pps / openmc_pps
    display(Markdown(f"**Performance:** OpenMC: {openmc_pps:,.0f} particles/s | YAMC: {yamc_pps:,.0f} particles/s | Speedup: {speedup:.2f}x"))

    # Neutron flux spectra plot
    n_fig = create_comparison_plot(
        openmc_results['n_energy_bins'],
        openmc_results['n_spectrum_mean'], openmc_results['n_spectrum_std'],
        yamc_results['n_spectrum_mean'], yamc_results['n_spectrum_std'],
        title_prefix='Neutron Flux'
    )
    n_fig.update_layout(title=f"{nuclide} - Neutron Flux Spectra Comparison")
    n_fig.show()

    # Neutron spectral agreement
    display(Markdown("### Neutron Spectral Agreement"))
    n_stats = calculate_spectral_agreement(
        openmc_results['n_spectrum_mean'], yamc_results['n_spectrum_mean'],
        openmc_results['n_spectrum_std'], yamc_results['n_spectrum_std'],
    )
    n_agreement_table = create_agreement_table(n_stats)
    display(n_agreement_table.style.hide(axis='index'))

    # Neutron tally table
    display(Markdown("### Neutron Tally Comparison"))
    n_table = create_neutron_tally_table(openmc_results, yamc_results)
    display(n_table.style.hide(axis='index'))

    # Photon flux spectra plot
    p_fig = create_comparison_plot(
        openmc_results['p_energy_bins'],
        openmc_results['p_spectrum_mean'], openmc_results['p_spectrum_std'],
        yamc_results['p_spectrum_mean'], yamc_results['p_spectrum_std'],
        title_prefix='Photon Flux'
    )
    p_fig.update_layout(title=f"{nuclide} ({element}) - Photon Flux Spectra Comparison")
    p_fig.show()

    # Photon spectral agreement
    display(Markdown("### Photon Spectral Agreement"))
    p_stats = calculate_spectral_agreement(
        openmc_results['p_spectrum_mean'], yamc_results['p_spectrum_mean'],
        openmc_results['p_spectrum_std'], yamc_results['p_spectrum_std'],
    )
    p_agreement_table = create_agreement_table(p_stats)
    display(p_agreement_table.style.hide(axis='index'))

    # Photon tally table
    display(Markdown("### Photon Tally Comparison"))
    p_table = create_photon_tally_table(openmc_results, yamc_results)
    display(p_table.style.hide(axis='index'))

    print()  # Add spacing between nuclides

## Fe54 (Fe)

**Performance:** OpenMC: 2,229,643 particles/s | YAMC: 2,607,328 particles/s | Speedup: 1.17x

### Neutron Spectral Agreement

Metric,Value,Note
Reduced Chi-Squared,1.08,"(ideal ~ 1.0, dof=134)"
Weighted Avg Rel. Diff,0.15%,(flux-weighted)
Median Rel. Diff,6.53%,(robust to outliers)
Agreement Quality,Excellent,(123/175 bins analyzed)


### Neutron Tally Comparison

MT,Reaction,OpenMC,YAMC,Rel. Diff (%)
1,"(n,total)",4.3518e-02 +/- 2.56e-05,4.3499e-02 +/- 1.57e-05,-0.04%
2,"(n,elastic)",1.9610e-02 +/- 1.14e-05,1.9603e-02 +/- 6.71e-06,-0.03%
3,nonelastic,nan +/- nan,2.3896e-02 +/- 9.22e-06,N/A
4,"(n,level)",nan +/- nan,8.5449e-03 +/- 3.17e-06,N/A
5,"(n,misc)",9.3179e-03 +/- 5.59e-06,9.3116e-03 +/- 3.80e-06,-0.07%
16,"(n,2n)",2.3832e-05 +/- 1.44e-08,2.3815e-05 +/- 9.97e-09,-0.07%
27,"(n,absorption)",6.0184e-03 +/- 3.66e-06,6.0154e-03 +/- 2.30e-06,-0.05%
51,"(n,n1)",6.6229e-04 +/- 3.55e-07,6.6258e-04 +/- 2.80e-07,0.04%
52,"(n,n2)",8.8949e-05 +/- 6.83e-08,8.9003e-05 +/- 6.07e-08,0.06%
53,"(n,n3)",4.1632e-05 +/- 4.17e-08,4.1679e-05 +/- 4.30e-08,0.11%


... (40 outputs left out)

### Photon Spectral Agreement

Metric,Value,Note
Reduced Chi-Squared,1.14,"(ideal ~ 1.0, dof=41)"
Weighted Avg Rel. Diff,1.48%,(flux-weighted)
Median Rel. Diff,1.81%,(robust to outliers)
Agreement Quality,Excellent,(40/42 bins analyzed)


### Photon Tally Comparison

Score,Reaction,OpenMC,YAMC,Rel. Diff (%)
coherent-scatter,Coherent (Rayleigh) scatter (MT 502),3.1023e-04 +/- 9.51e-07,3.0928e-04 +/- 2.15e-06,-0.31%
flux,Total photon flux,6.4002e-02 +/- 1.32e-04,6.3884e-02 +/- 1.08e-04,-0.18%
heating,Heating (KERMA),3.4812e+00 +/- 6.41e-02,3.5130e+00 +/- 4.48e-02,0.91%
incoherent-scatter,Incoherent (Compton) scatter (MT 504),4.0856e-03 +/- 8.36e-06,4.0707e-03 +/- 8.81e-06,-0.36%
pair-production,Pair production (MT 516),8.8271e-05 +/- 5.62e-07,8.9074e-05 +/- 4.20e-07,0.91%
photoelectric,Photoelectric absorption (MT 522),2.8435e-03 +/- 2.35e-05,2.8032e-03 +/- 2.80e-05,-1.42%
